# Therm-FM: Few-Shot Fine-Tuning Across Geometries, geometry1 -> geometry3

This pretrains a `CNOFNOHybrid` on `geometry1` (10mm x 10mm, 4 power blocks), then
few-shot fine-tunes it on `geometry3` (25mm x 25mm, 8 power blocks): a different physical
floorplan and power pattern, but the same mesh resolution (100x100x40), which
`CNOFNOHybrid`'s architecture requires (its encoder/decoder conv strides are built for one
fixed grid shape; genuinely cross-grid-shape pretraining needs the separate
adaptive-pooling `PI-CNO-DeepONet` architecture in `src/deeponet/cno_model.py`, out of
scope here).

Compares fine-tuned-from-pretrained vs trained-from-scratch at several shot counts (the
actually meaningful comparison: does pretraining help at a given data budget), and runs
weight-drift analysis on the best fine-tuned checkpoint to show what the model actually
changed to adapt to the new geometry.

## Kaggle Dataset Setup

1. Source code: upload `src/` (slug suggestion: `thermo-pinn-src`)
2. 3D-ICE training data: upload `.npz` files from `data/3d-ice/` (needs both geometry1 and
   geometry3; the full `3d-ice` dataset covers both already)

Expected time: pretrain about 15-25 min (geometry1, 400 epochs) plus a shot sweep (several
short fine-tunes, a few minutes each), roughly 30-45 min total on a T4.

Note on checkpoint format: `FNOTrainer` (used here for pretraining) saves checkpoints
under the key `'model_state'`; this repo's `finetune_therm_fm.py` saves under `'model'`.
All loading in this notebook accepts either key.


In [ ]:
import subprocess, sys

# Install any missing packages (PyYAML is the only non-standard dep)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import os
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import sys
from pathlib import Path

# ── Kaggle dataset paths ────────────────────────────────────────────────────
# Adjust these slugs to match the datasets you attached to this notebook.
SRC_DATASET   = 'thermo-pinn-src'    # dataset containing the src/ folder
DATA_DATASET  = '3dice-thermal-data' # dataset containing the .npz files

SRC_ROOT  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_ROOT = Path(f'/kaggle/input/{DATA_DATASET}')
OUT_DIR   = Path('/kaggle/working/checkpoints/therm_fm')

# Add source root to path so we can import src.*
sys.path.insert(0, str(SRC_ROOT))

# Verify
assert SRC_ROOT.exists(),  f'Source dataset not found at {SRC_ROOT}. Check SRC_DATASET slug.'
assert DATA_ROOT.exists(), f'Data dataset not found at {DATA_ROOT}. Check DATA_DATASET slug.'
print('Source root:', SRC_ROOT)
print('Data root:  ', DATA_ROOT)

In [ ]:
import logging, json, math
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import NormStats, compute_norm_stats
from src.fno.model import build_cno_fno
from src.fno.data_loader import FNODataset
from src.fno.trainer import FNOTrainer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)


# ── Inline: freeze/unfreeze helpers (mirrors scripts/finetune_therm_fm.py) ────
# CNOFNOHybrid's actual top-level parameter groups (verified via named_parameters()):
#   lift, enc_res, enc_down   -- CNN encoder
#   latent_blocks             -- FiLM-FNO blocks at the pooled latent resolution
#   film_gen                  -- FiLM generator (BCs -> per-block gamma/beta)
#   dec_res, dec_fuse         -- CNN decoder
#   proj                      -- final projection head
_ENCODER_PREFIXES = ('lift', 'enc_res', 'enc_down')
_DECODER_PREFIXES = ('dec_res', 'dec_fuse')
_TAIL_BLOCK_ATTR  = 'latent_blocks'

def freeze_encoder(model):
    frozen = 0
    for name, param in model.named_parameters():
        if name.startswith(_ENCODER_PREFIXES):
            param.requires_grad_(False)
            frozen += param.numel()
    return frozen

def unfreeze_film_and_tail(model, n_tail_blocks=2):
    unfrozen = 0
    n_latent = len(getattr(model, _TAIL_BLOCK_ATTR))
    tail_start = max(0, n_latent - n_tail_blocks)
    for name, param in model.named_parameters():
        is_film = name.startswith('film_gen')
        is_decoder = name.startswith(_DECODER_PREFIXES)
        is_proj = name.startswith('proj')
        is_tail = any(name.startswith(f'{_TAIL_BLOCK_ATTR}.{i}') for i in range(tail_start, n_latent))
        if is_film or is_decoder or is_proj or is_tail:
            param.requires_grad_(True)
            unfrozen += param.numel()
        else:
            param.requires_grad_(False)
    return unfrozen

print('Imports + freeze/unfreeze helpers OK')

In [ ]:
SOURCE_GEOM   = 'geometry1'
HELD_OUT_GEOM = 'geometry3'
CHANNELS      = 32
N_BLOCKS      = 4
PRETRAIN_EPOCHS = 400
SHOT_COUNTS     = [5, 10, 20]
FINETUNE_EPOCHS = 60
SCRATCH_EPOCHS  = 60      # same epoch budget as fine-tune, for a fair comparison
LR_PRETRAIN     = 1e-3
LR_FINETUNE     = 1e-4    # lower LR for fine-tuning, matches finetune_therm_fm.py default
BATCH_SIZE      = 4
N_TAIL_BLOCKS   = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
print(f'Source: {SOURCE_GEOM}  ->  Held-out: {HELD_OUT_GEOM}')

In [ ]:
source_geometry = get_geometry_by_name(SOURCE_GEOM)
held_out_geometry = get_geometry_by_name(HELD_OUT_GEOM)

assert source_geometry.mesh_resolution == held_out_geometry.mesh_resolution, (
    f'CNOFNOHybrid requires matching grid shape: {source_geometry.mesh_resolution} vs '
    f'{held_out_geometry.mesh_resolution}. Pick a different SOURCE_GEOM/HELD_OUT_GEOM pair '
    f'that share a mesh_resolution.'
)
grid_shape = source_geometry.mesh_resolution
print('Source geometry:'); print(source_geometry.summary())
print('Held-out geometry:'); print(held_out_geometry.summary())
print('Shared grid shape:', grid_shape)

In [ ]:
def collect_files(data_dir: Path, geom: str, split: str):
    files = sorted(data_dir.rglob(f'{geom}_{split}_*.npz'))
    if not files:
        files = sorted(data_dir.glob(f'{geom}_{split}_*.npz'))
    return files

In [ ]:
source_train_files  = collect_files(DATA_ROOT, SOURCE_GEOM, 'train')
source_test_files   = collect_files(DATA_ROOT, SOURCE_GEOM, 'test')
heldout_train_files = collect_files(DATA_ROOT, HELD_OUT_GEOM, 'train')
heldout_test_files  = collect_files(DATA_ROOT, HELD_OUT_GEOM, 'test')

assert source_train_files,  f'No {SOURCE_GEOM} training files found in {DATA_ROOT}'
assert heldout_train_files, f'No {HELD_OUT_GEOM} training files found in {DATA_ROOT}'
print(f'{SOURCE_GEOM}: {len(source_train_files)} train, {len(source_test_files)} test')
print(f'{HELD_OUT_GEOM}: {len(heldout_train_files)} train, {len(heldout_test_files)} test')

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
norm_path = OUT_DIR / 'norm_stats.json'

if norm_path.exists():
    norm_stats = NormStats.load(norm_path)
    print('Loaded existing norm stats from', norm_path)
else:
    print(f'Computing norm stats over {SOURCE_GEOM} training files (source-only, as a foundation model would see)...')
    norm_stats = compute_norm_stats(source_train_files, {SOURCE_GEOM: source_geometry})
    norm_stats.save(norm_path)
    print('Saved norm stats to', norm_path)

print(f'T range: [{norm_stats.T_min:.1f}, {norm_stats.T_max:.1f}] K')

In [ ]:
print('Loading source datasets...')
source_train_ds = FNODataset(source_train_files, norm_stats, grid_shape)
source_val_ds   = FNODataset(source_test_files or source_train_files[-3:], norm_stats, grid_shape)
print(f'Source train: {len(source_train_ds)}  val: {len(source_val_ds)}')

In [ ]:
pretrain_dir = OUT_DIR / 'pretrain'
pretrain_ckpt_path = pretrain_dir / f'{SOURCE_GEOM}_pretrain_best.pt'

if pretrain_ckpt_path.exists():
    print('Found existing pretrained checkpoint, skipping pretrain:', pretrain_ckpt_path)
    pretrained_model = build_cno_fno(grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS, device=DEVICE)
    ckpt = torch.load(pretrain_ckpt_path, map_location=DEVICE)
    pretrained_model.load_state_dict(ckpt.get('model') or ckpt.get('model_state') or ckpt)
else:
    pretrained_model = build_cno_fno(grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS, device=DEVICE)
    print(f'CNO-FNO: {pretrained_model.n_parameters:,} params')
    pretrain_trainer = FNOTrainer(
        model=pretrained_model, norm_stats=norm_stats,
        train_data=source_train_ds, val_data=source_val_ds,
        output_dir=pretrain_dir, batch_size=BATCH_SIZE, epochs=PRETRAIN_EPOCHS, lr=LR_PRETRAIN,
        device=DEVICE, geometry_name=f'{SOURCE_GEOM}_pretrain', geometry=source_geometry,
    )
    pretrain_ckpt_path = pretrain_trainer.train()
    print('Pretrain done. Best val MAE:', pretrain_trainer.best_val_mae)

pretrained_state = {k: v.clone() for k, v in pretrained_model.state_dict().items()}
print('Pretrained state captured for later weight-drift comparison')

In [ ]:
heldout_val_ds = FNODataset(heldout_test_files or heldout_train_files[-3:], norm_stats, grid_shape)
print(f'Held-out val: {len(heldout_val_ds)} scenarios (fixed across all shot counts)')

In [ ]:
def finetune_from_pretrained(shots, epochs=FINETUNE_EPOCHS, seed=0):
    torch.manual_seed(seed)
    model = build_cno_fno(grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS, device=DEVICE)
    model.load_state_dict(pretrained_state)

    freeze_encoder(model)
    unfreeze_film_and_tail(model, n_tail_blocks=N_TAIL_BLOCKS)

    shot_files = heldout_train_files[:shots]
    train_ds = FNODataset(shot_files, norm_stats, grid_shape)

    trainer = FNOTrainer(
        model=model, norm_stats=norm_stats, train_data=train_ds, val_data=heldout_val_ds,
        output_dir=OUT_DIR / f'finetune_{shots}shot', batch_size=min(BATCH_SIZE, shots),
        epochs=epochs, lr=LR_FINETUNE, device=DEVICE,
        geometry_name=f'{HELD_OUT_GEOM}_finetune_{shots}shot', geometry=held_out_geometry,
    )
    trainer.train()
    return trainer.best_val_mae, model


def train_from_scratch(shots, epochs=SCRATCH_EPOCHS, seed=0):
    torch.manual_seed(seed)
    model = build_cno_fno(grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS, device=DEVICE)

    shot_files = heldout_train_files[:shots]
    train_ds = FNODataset(shot_files, norm_stats, grid_shape)

    trainer = FNOTrainer(
        model=model, norm_stats=norm_stats, train_data=train_ds, val_data=heldout_val_ds,
        output_dir=OUT_DIR / f'scratch_{shots}shot', batch_size=min(BATCH_SIZE, shots),
        epochs=epochs, lr=LR_PRETRAIN, device=DEVICE,
        geometry_name=f'{HELD_OUT_GEOM}_scratch_{shots}shot', geometry=held_out_geometry,
    )
    trainer.train()
    return trainer.best_val_mae, model


shots_results = {'finetuned': {}, 'scratch': {}}
best_finetuned_model = None
best_finetuned_mae = float('inf')

for shots in SHOT_COUNTS:
    print(f'\n=== {shots}-shot ===')
    mae_ft, model_ft = finetune_from_pretrained(shots)
    mae_sc, _        = train_from_scratch(shots)
    shots_results['finetuned'][shots] = mae_ft
    shots_results['scratch'][shots]   = mae_sc
    print(f'  fine-tuned: {mae_ft:.3f} K   scratch: {mae_sc:.3f} K')
    if mae_ft < best_finetuned_mae:
        best_finetuned_mae = mae_ft
        best_finetuned_model = model_ft

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
shots_sorted = sorted(shots_results['finetuned'].keys())
ft_vals = [shots_results['finetuned'][s] for s in shots_sorted]
sc_vals = [shots_results['scratch'][s] for s in shots_sorted]
ax.plot(shots_sorted, ft_vals, marker='o', label='Fine-tuned from pretrained', color='steelblue')
ax.plot(shots_sorted, sc_vals, marker='o', label='Trained from scratch', color='tomato')
ax.set_xlabel('Number of training shots'); ax.set_ylabel('Val MAE (K)')
ax.set_title(f'Therm-FM: {SOURCE_GEOM} -> {HELD_OUT_GEOM}  few-shot transfer')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shots_vs_mae.png', dpi=150, bbox_inches='tight')
plt.show()

print('Where scratch < fine-tuned, pretraining does not help at that shot count.')
print('Where fine-tuned < scratch, pretraining is paying off.')

In [ ]:
_NEAR_ZERO_INIT_FLOOR = 1e-4

def _group_for(name):
    if name.startswith(_ENCODER_PREFIXES): return 'encoder (frozen)'
    if name.startswith('film_gen'): return 'film_gen'
    if name.startswith(_DECODER_PREFIXES): return 'decoder'
    if name.startswith('proj'): return 'proj'
    if name.startswith(_TAIL_BLOCK_ATTR): return 'latent_blocks (tail + frozen-middle mixed)'
    return 'other'

def compute_weight_drift(pretrained_sd, finetuned_sd):
    records = []
    for key in sorted(set(pretrained_sd) & set(finetuned_sd)):
        w0, w1 = pretrained_sd[key], finetuned_sd[key]
        if not w0.is_complex(): w0 = w0.float()
        if not w1.is_complex(): w1 = w1.float()
        if w0.shape != w1.shape:
            continue
        delta_norm = (w1 - w0).norm().item()
        base_norm = w0.norm().item()
        near_zero = base_norm < _NEAR_ZERO_INIT_FLOOR
        rel_drift = float('nan') if near_zero else delta_norm / base_norm
        records.append({'name': key, 'group': _group_for(key), 'n_params': w0.numel(),
                         'delta_norm': delta_norm, 'base_norm': base_norm,
                         'rel_drift': rel_drift, 'near_zero_init': near_zero})
    return records

def summarize_by_group(records):
    groups = {}
    for r in records:
        groups.setdefault(r['group'], []).append(r)
    summary = {}
    for group, recs in groups.items():
        total_params = sum(r['n_params'] for r in recs)
        total_delta = sum(r['delta_norm'] for r in recs)
        finite = [r for r in recs if not r['near_zero_init']]
        fp = sum(r['n_params'] for r in finite)
        mean_rel = sum(r['rel_drift'] * r['n_params'] for r in finite) / fp if fp > 0 else float('nan')
        summary[group] = {'n_tensors': len(recs), 'n_params': total_params,
                           'total_abs_drift_norm': total_delta, 'mean_rel_drift': mean_rel}
    return summary

finetuned_state = {k: v.clone() for k, v in best_finetuned_model.state_dict().items()}
drift_records = compute_weight_drift(pretrained_state, finetuned_state)
drift_summary = summarize_by_group(drift_records)

print(f'{"Group":<45} {"n_params":>10} {"abs_drift":>12} {"mean_rel_drift":>16}')
print('-' * 85)
for group, stats in sorted(drift_summary.items(), key=lambda kv: -kv[1]['total_abs_drift_norm']):
    rel_str = 'n/a' if math.isnan(stats['mean_rel_drift']) else f"{stats['mean_rel_drift']:.4f}"
    print(f'{group:<45} {stats["n_params"]:>10} {stats["total_abs_drift_norm"]:>12.4f} {rel_str:>16}')

encoder_drift = drift_summary.get('encoder (frozen)', {}).get('total_abs_drift_norm', 0.0)
print(f'\nEncoder drift: {encoder_drift:.6f} (should be exactly 0.0 -- confirms freeze_encoder worked)')

groups_sorted = sorted(drift_summary.keys(), key=lambda g: -drift_summary[g]['total_abs_drift_norm'])
abs_vals = [drift_summary[g]['total_abs_drift_norm'] for g in groups_sorted]
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(len(groups_sorted)), abs_vals, color='tab:blue')
ax.set_xticks(range(len(groups_sorted))); ax.set_xticklabels(groups_sorted, rotation=25, ha='right')
ax.set_ylabel('sum ||W_finetuned - W_pretrained||'); ax.set_title(f'Weight drift by group (best fine-tune, {HELD_OUT_GEOM})')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'weight_drift.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
summary = {
    'source_geometry': SOURCE_GEOM,
    'held_out_geometry': HELD_OUT_GEOM,
    'grid_shape': list(grid_shape),
    'pretrain_epochs': PRETRAIN_EPOCHS,
    'shot_counts': SHOT_COUNTS,
    'finetune_epochs': FINETUNE_EPOCHS,
    'scratch_epochs': SCRATCH_EPOCHS,
    'shots_vs_mae': shots_results,
    'best_finetuned_mae_K': best_finetuned_mae,
    'weight_drift_by_group': drift_summary,
}

with open(OUT_DIR / 'therm_fm_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(json.dumps(summary, indent=2, default=str))